<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/20_regularization/20_1_Practical_Training/20_1_9_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regularization: Exercise
## The Regularization Toolkit on Wine Quality
---

## Overview

In notebook 19_1_9 you built a `WineNet` to classify red wines into Low / Medium / High quality, and found it was *not* meaningfully better than XGBoost on this small (~1,600-sample) dataset. Now you have a full regularization toolkit — dropout, batch normalization, learning rate schedules, and early stopping with checkpointing. This exercise asks a direct question: **does that toolkit change the verdict?**

You will rebuild `WineNet`, add each technique in turn, and judge the effect by reading the loss and macro-F1 curves. As in the other exercises, each task gives you a scaffold to fill in; run the matching **"Execute to see solution"** cell only after you have attempted it yourself.

The dataset and the three-class target (Low: quality ≤ 5, Medium: 6, High: ≥ 7) are exactly as in 19_1_9 — the *only* thing changing across this notebook is the model and how it is trained.

**Reminder of the prior knowledge you are applying:**
- `nn.Dropout(p)` after the activation; `model.train()`/`model.eval()` matter (20_1_1)
- `nn.BatchNorm1d` as `Linear → BatchNorm → ReLU` (20_1_2)
- `CosineAnnealingLR`, with `scheduler.step()` after `optimizer.step()`, once per epoch (20_1_3)
- early stopping: checkpoint on new best val loss, restore at the end (20_1_4)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import compute_class_weight
import tempfile, os

import seaborn as sns
sns.set_theme(style='whitegrid')
torch.manual_seed(42)

class_names = ['Low', 'Medium', 'High']

---

### Task 1 — Load the data and build the splits

Load the red-wine dataset, recreate the three-class `quality_group` target, integer-encode it, and produce the standard 60/20/20 train/val/test split with a `StandardScaler` fit on training data only. This is the same setup as 19_1_9; reproduce it so the rest of the notebook has data to work with.

In [2]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

# TODO: read the CSV (sep=';'), build the 3-class target, split, scale, tensorize
# wine = pd.read_csv(url, sep=';')
# def assign_group(q): ...
# wine['quality_group'] = wine['quality'].apply(assign_group)
# ... label_map, X_raw, y_raw, train_test_split x2, StandardScaler, torch.tensor ...
# BATCH_SIZE = 32
# train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

In [3]:
# @title Execute to see solution
solution = '''
wine = pd.read_csv(url, sep=";")

def assign_group(q):
    if q <= 5:   return "Low"
    elif q == 6: return "Medium"
    else:        return "High"

wine["quality_group"] = wine["quality"].apply(assign_group)

label_map = {name: i for i, name in enumerate(class_names)}
y_raw = wine["quality_group"].map(label_map).to_numpy(dtype=np.int64)
X_raw = wine.drop(columns=["quality", "quality_group"]).to_numpy(dtype=np.float32)

X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

scaler     = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

X_train = torch.tensor(X_train_np); y_train = torch.tensor(y_train_np)
X_val   = torch.tensor(X_val_np);   y_val   = torch.tensor(y_val_np)
X_test  = torch.tensor(X_test_np);  y_test  = torch.tensor(y_test_np)

# balanced class weights for the imbalanced target
class_weights = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(3), y=y_train_np),
    dtype=torch.float32)

BATCH_SIZE   = 32
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

print(f"Train {X_train.shape[0]}  Val {X_val.shape[0]}  Test {X_test.shape[0]}  Features {X_train.shape[1]}")
print("Class counts:", np.bincount(y_raw).tolist())
'''
print(solution)


wine = pd.read_csv(url, sep=";")

def assign_group(q):
    if q <= 5:   return "Low"
    elif q == 6: return "Medium"
    else:        return "High"

wine["quality_group"] = wine["quality"].apply(assign_group)

label_map = {name: i for i, name in enumerate(class_names)}
y_raw = wine["quality_group"].map(label_map).to_numpy(dtype=np.int64)
X_raw = wine.drop(columns=["quality", "quality_group"]).to_numpy(dtype=np.float32)

X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

scaler     = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

X_train = torch.tensor(X_train_np); y_train = torch.tensor(y_train_np)
X_val   = torch.tensor(X_val_np);   y_val   = torch.tensor(y_val_np)
X_test  = torch.tensor(

---

### Task 2 — A reusable training function, and the dropout question

You will train several variants, so first write **one** `train_model()` function that you can reuse — it should run the training loop, track train loss and validation macro F1 per epoch, and support an optional scheduler and optional early stopping with checkpointing (return the model restored to its best epoch).

Then use it to compare the plain `WineNet` from 19_1_9 against the *same* network with `nn.Dropout(0.3)` after each ReLU. Does dropout help on this dataset?

> Tip: `WineNet` is `11 → 64 → 32 → 3` with `CrossEntropyLoss` (no softmax in the model). Place dropout as `Linear → ReLU → Dropout`.

In [4]:
# TODO: write train_model(model, optimizer, criterion, max_epochs, scheduler=None, patience=None)
# TODO: define a build_winenet(use_bn, use_dropout) helper
# TODO: train baseline vs +dropout, print test macro F1 of each

In [5]:
# @title Execute to see solution
solution = '''
def build_winenet(use_bn=False, use_dropout=False, p=0.3):
    torch.manual_seed(42)
    layers, in_dim = [], 11
    for width in (64, 32):
        layers.append(nn.Linear(in_dim, width))
        if use_bn:      layers.append(nn.BatchNorm1d(width))
        layers.append(nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(p))
        in_dim = width
    layers.append(nn.Linear(in_dim, 3))   # logits
    return nn.Sequential(*layers)


def train_model(model, optimizer, criterion, max_epochs=200,
                scheduler=None, patience=None):
    ckpt = os.path.join(tempfile.mkdtemp(), "best.pt")
    best_val, best_epoch, pc = float("inf"), -1, 0
    stopped_at = max_epochs - 1
    train_loss, val_f1 = [], []
    for epoch in range(max_epochs):
        model.train()
        running = 0.0
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward(); optimizer.step()
            running += loss.item()
        train_loss.append(running / len(train_loader))

        model.eval()
        with torch.no_grad():
            v_loss = criterion(model(X_val), y_val).item()
            v_pred = model(X_val).argmax(dim=1).numpy()
        val_f1.append(f1_score(y_val.numpy(), v_pred, average="macro"))

        if v_loss < best_val:
            best_val, best_epoch, pc = v_loss, epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            pc += 1
        if patience is not None and pc >= patience:
            stopped_at = epoch; break
        if scheduler is not None:
            scheduler.step()

    model.load_state_dict(torch.load(ckpt, weights_only=True))
    return model, {"train_loss": train_loss, "val_f1": val_f1,
                   "best_epoch": best_epoch, "stopped_at": stopped_at}


crit = nn.CrossEntropyLoss(weight=class_weights)

base = build_winenet()
base, base_h = train_model(base, torch.optim.Adam(base.parameters(), lr=1e-3, weight_decay=1e-4), crit)

drop = build_winenet(use_dropout=True)
drop, drop_h = train_model(drop, torch.optim.Adam(drop.parameters(), lr=1e-3, weight_decay=1e-4), crit)

for name, m in [("baseline", base), ("+ dropout", drop)]:
    m.eval()
    with torch.no_grad():
        preds = m(X_test).argmax(dim=1).numpy()
    print(f"{name:12} test macro F1: {f1_score(y_test.numpy(), preds, average='macro'):.3f}")
'''
print(solution)


def build_winenet(use_bn=False, use_dropout=False, p=0.3):
    torch.manual_seed(42)
    layers, in_dim = [], 11
    for width in (64, 32):
        layers.append(nn.Linear(in_dim, width))
        if use_bn:      layers.append(nn.BatchNorm1d(width))
        layers.append(nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(p))
        in_dim = width
    layers.append(nn.Linear(in_dim, 3))   # logits
    return nn.Sequential(*layers)


def train_model(model, optimizer, criterion, max_epochs=200,
                scheduler=None, patience=None):
    ckpt = os.path.join(tempfile.mkdtemp(), "best.pt")
    best_val, best_epoch, pc = float("inf"), -1, 0
    stopped_at = max_epochs - 1
    train_loss, val_f1 = [], []
    for epoch in range(max_epochs):
        model.train()
        running = 0.0
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward(); optimizer.step()
            running += loss.ite

---

### Task 3 — Add batch normalization

Use your `build_winenet(use_bn=True, ...)` switch to train a version with `nn.BatchNorm1d` in the standard `Linear → BatchNorm → ReLU` order. Compare its validation macro-F1 curve against the baseline's. Does convergence look any faster or steadier? (Recall from 20_1_2 that on a small, easy problem the effect can be slight.)

In [6]:
# TODO: train a BatchNorm WineNet; plot its val_f1 curve against the baseline's

In [7]:
# @title Execute to see solution
solution = '''
bn = build_winenet(use_bn=True)
bn, bn_h = train_model(bn, torch.optim.Adam(bn.parameters(), lr=1e-3, weight_decay=1e-4), crit)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(base_h["val_f1"], color="#C0392B",   lw=2, label="baseline")
ax.plot(bn_h["val_f1"],   color="steelblue", lw=2, label="with BatchNorm")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation macro F1")
ax.set_title("Does BatchNorm change convergence on Wine?")
ax.legend(); plt.tight_layout(); plt.show()

bn.eval()
with torch.no_grad():
    preds = bn(X_test).argmax(dim=1).numpy()
print(f"BatchNorm test macro F1: {f1_score(y_test.numpy(), preds, average='macro'):.3f}")
'''
print(solution)


bn = build_winenet(use_bn=True)
bn, bn_h = train_model(bn, torch.optim.Adam(bn.parameters(), lr=1e-3, weight_decay=1e-4), crit)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(base_h["val_f1"], color="#C0392B",   lw=2, label="baseline")
ax.plot(bn_h["val_f1"],   color="steelblue", lw=2, label="with BatchNorm")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation macro F1")
ax.set_title("Does BatchNorm change convergence on Wine?")
ax.legend(); plt.tight_layout(); plt.show()

bn.eval()
with torch.no_grad():
    preds = bn(X_test).argmax(dim=1).numpy()
print(f"BatchNorm test macro F1: {f1_score(y_test.numpy(), preds, average='macro'):.3f}")



---

### Task 4 — Apply a cosine learning rate schedule

Train a WineNet with a `CosineAnnealingLR` schedule (`T_max` equal to your epoch count). Pass it to `train_model` via the `scheduler` argument. Does the final validation macro F1 improve over a fixed learning rate?

In [8]:
# TODO: train with CosineAnnealingLR; report final/best val macro F1

In [9]:
# @title Execute to see solution
solution = '''
EPOCHS = 200
sched_model = build_winenet()
opt = torch.optim.Adam(sched_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched_model, sched_h = train_model(
    sched_model, opt, crit, max_epochs=EPOCHS,
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS))

print(f"fixed-LR baseline  best val macro F1: {max(base_h['val_f1']):.3f}")
print(f"cosine schedule    best val macro F1: {max(sched_h['val_f1']):.3f}")

sched_model.eval()
with torch.no_grad():
    preds = sched_model(X_test).argmax(dim=1).numpy()
print(f"cosine schedule    test macro F1: {f1_score(y_test.numpy(), preds, average='macro'):.3f}")
'''
print(solution)


EPOCHS = 200
sched_model = build_winenet()
opt = torch.optim.Adam(sched_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched_model, sched_h = train_model(
    sched_model, opt, crit, max_epochs=EPOCHS,
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS))

print(f"fixed-LR baseline  best val macro F1: {max(base_h['val_f1']):.3f}")
print(f"cosine schedule    best val macro F1: {max(sched_h['val_f1']):.3f}")

sched_model.eval()
with torch.no_grad():
    preds = sched_model(X_test).argmax(dim=1).numpy()
print(f"cosine schedule    test macro F1: {f1_score(y_test.numpy(), preds, average='macro'):.3f}")



---

### Task 5 — Early stopping with checkpointing

Train a WineNet with early stopping by passing `patience` (e.g. 25) to `train_model`. Report which epoch held the best checkpoint and which epoch training actually stopped at. The gap between them is the wasted tail your `patience` window allowed.

In [10]:
# TODO: train with patience=25; print best_epoch and stopped_at from the history

In [11]:
# @title Execute to see solution
solution = '''
es_model = build_winenet()
es_model, es_h = train_model(
    es_model, torch.optim.Adam(es_model.parameters(), lr=1e-3, weight_decay=1e-4),
    crit, max_epochs=200, patience=25)

print(f"Best checkpoint at epoch : {es_h['best_epoch']}")
print(f"Training stopped at epoch: {es_h['stopped_at']}")
print(f"Wasted tail (patience)   : {es_h['stopped_at'] - es_h['best_epoch']} epochs")
'''
print(solution)


es_model = build_winenet()
es_model, es_h = train_model(
    es_model, torch.optim.Adam(es_model.parameters(), lr=1e-3, weight_decay=1e-4),
    crit, max_epochs=200, patience=25)

print(f"Best checkpoint at epoch : {es_h['best_epoch']}")
print(f"Training stopped at epoch: {es_h['stopped_at']}")
print(f"Wasted tail (patience)   : {es_h['stopped_at'] - es_h['best_epoch']} epochs")



---

### Task 6 — The fully regularized WineNet

Put it all together: a WineNet with BatchNorm **and** dropout, trained with a cosine schedule **and** early stopping. Evaluate the restored best model on the held-out test set with a full `classification_report`, and compare its macro F1 to the plain baseline from Task 2.

In [12]:
# TODO: build_winenet(use_bn=True, use_dropout=True); train with scheduler + patience;
#       classification_report on X_test; compare macro F1 to baseline

In [13]:
# @title Execute to see solution
solution = '''
EPOCHS = 200
full = build_winenet(use_bn=True, use_dropout=True)
opt  = torch.optim.Adam(full.parameters(), lr=1e-3, weight_decay=1e-4)
full, full_h = train_model(
    full, opt, crit, max_epochs=EPOCHS,
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS),
    patience=25)

full.eval()
with torch.no_grad():
    preds = full(X_test).argmax(dim=1).numpy()

print(f"Fully regularized WineNet — best epoch {full_h['best_epoch']}, stopped @ {full_h['stopped_at']}")
print(classification_report(y_test.numpy(), preds, target_names=class_names, digits=3))

base.eval()
with torch.no_grad():
    base_preds = base(X_test).argmax(dim=1).numpy()
print(f"baseline           test macro F1: {f1_score(y_test.numpy(), base_preds, average='macro'):.3f}")
print(f"fully regularized  test macro F1: {f1_score(y_test.numpy(), preds,      average='macro'):.3f}")
'''
print(solution)


EPOCHS = 200
full = build_winenet(use_bn=True, use_dropout=True)
opt  = torch.optim.Adam(full.parameters(), lr=1e-3, weight_decay=1e-4)
full, full_h = train_model(
    full, opt, crit, max_epochs=EPOCHS,
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS),
    patience=25)

full.eval()
with torch.no_grad():
    preds = full(X_test).argmax(dim=1).numpy()

print(f"Fully regularized WineNet — best epoch {full_h['best_epoch']}, stopped @ {full_h['stopped_at']}")
print(classification_report(y_test.numpy(), preds, target_names=class_names, digits=3))

base.eval()
with torch.no_grad():
    base_preds = base(X_test).argmax(dim=1).numpy()
print(f"baseline           test macro F1: {f1_score(y_test.numpy(), base_preds, average='macro'):.3f}")
print(f"fully regularized  test macro F1: {f1_score(y_test.numpy(), preds,      average='macro'):.3f}")



---

## Reflection

Answer in the markdown cells below, using the numbers and curves your runs produced.

**Q1.** The wine dataset has only ~1,600 samples. Which regularization technique had the most visible impact on the loss / macro-F1 curves, and which had little or none? Why might that be — what does it suggest about whether this network was overfitting in the first place?

**Q2.** At what epoch did early stopping trigger (Task 5), and how far was that past the best checkpoint? What does that gap tell you about how much of the training run was actually useful?

**Q3.** After applying the whole toolkit, is the neural network now clearly better than XGBoost on this dataset (recall from 19_1_9 that they were comparable)? What does your answer suggest about *when* a neural network is the right tool — and when a gradient-boosted tree is the more sensible default?

*Write your answer to Q1 here.*

*Write your answer to Q2 here.*

*Write your answer to Q3 here.*

---

## Wrap-Up

This exercise closes module 20. You took the same `WineNet` from 19_1_9 and applied the full practical-training toolkit — dropout, batch normalization, a cosine learning rate schedule, and early stopping with checkpointing — and most likely found that on a small, noisy dataset like this the techniques did **not** dramatically improve the result. That is the honest and important finding from 20_1_5, confirmed by your own hands: regularization is a fix for overfitting, and when a model is not overfitting (or when the dataset is simply too small and noisy for a neural network to shine), the toolkit keeps training clean and stable without conjuring accuracy that the data cannot support. Knowing that — and reaching for a gradient-boosted tree when it is the better fit — is as much a part of practical machine learning as the techniques themselves.